# 06 - Retriever Analytics

This notebook reconstructs the bridge between the canonical corpora, the specialized retriever, the generic baseline, and the application corpus. It prepares the indexed document universe, mirrored query bank, and query-level analytics used in the downstream stages.

## Methodological role

This stage turns the trained retrieval infrastructure into application-ready evidence bundles and gap candidates.

In [ ]:
# ============================================================
# Instalacao de dependencias para Colab
# ============================================================
!pip install -U -q pip setuptools wheel
!pip uninstall -y -q numpy scipy scikit-learn sentence-transformers transformers faiss-cpu numba umap-learn hdbscan || true
!pip install -U -q numpy==2.0.2 scipy==1.14.1 pandas==2.2.2 scikit-learn==1.6.1 numba==0.60.0 openpyxl pyarrow jedi==0.19.2
!pip install -U -q faiss-cpu sentence-transformers==2.7.0 transformers==4.45.2

import numpy, scipy, sklearn, sentence_transformers, transformers
print("numpy            :", numpy.__version__)
print("scipy            :", scipy.__version__)
print("scikit-learn     :", sklearn.__version__)
print("sentence-transformers:", sentence_transformers.__version__)
print("transformers     :", transformers.__version__)

print("Dependencias instaladas.")


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import ast
import hashlib
import json
import math
import os
import re
import shutil
import time
from datetime import datetime
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 240)

print("Torch:", torch.__version__)
print("GPU disponivel:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# ============================================================
# Configuracao geral
# ============================================================

DRIVE_ROOT = Path("/content/drive/MyDrive/Unicamp")
PROJECT_ROOT = DRIVE_ROOT / "artigo bibliometria" / "grounded-scientometrics-solarphysics-retrieval"
DATA_ROOT = DRIVE_ROOT / "artigo bibliometria" / "base de dados" / "Artigo_Bibliometria Base Bruta" / "BASES_UNIFICADAS_POR_TEMA"

TARGET_CORPUS = "ML_Multimodal"
READ_STAGE_CONSOL = "00_consolidacao"
READ_STAGE_ABSTRACT = "01_abstract_llm"
WRITE_STAGE = "06_pipe2_retriever_analytics"
MODEL_ROOT = DATA_ROOT / "_cross_corpus_rebuild" / "05_scibert_solarphysics_search" / "checkpoints" / "SciBERT-SolarPhysics-Search"

PERIODS = ["core", "holdout"]
QUERY_CORPORA = ["Nucleo", "PIML", "CombFinal"]
GENERIC_MODEL_ID = "allenai/scibert_scivocab_uncased"
TOPICS_PER_CORPUS = 20
TOPK = 20
EMBED_BATCH_SIZE = 64

assert PROJECT_ROOT.exists(), f"PROJECT_ROOT nao encontrado: {PROJECT_ROOT}"
assert DATA_ROOT.exists(), f"DATA_ROOT nao encontrado: {DATA_ROOT}"
assert MODEL_ROOT.exists(), f"Modelo especializado nao encontrado: {MODEL_ROOT}"

print("MODEL_ROOT =", MODEL_ROOT)
print("TOPK =", TOPK)


In [ ]:
# ============================================================
# Saidas, logs e helpers
# ============================================================

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
PIPE_START_TS = time.time()


def ensure_dir(path: Path) -> Path:
    path.mkdir(parents=True, exist_ok=True)
    return path


WRITE_ROOT = ensure_dir(DATA_ROOT / TARGET_CORPUS / "04_rebuild_outputs" / WRITE_STAGE)
GLOBAL_LOG_DIR = ensure_dir(PROJECT_ROOT / "outputs" / "camada2_logs")
GLOBAL_LOG_FILE = GLOBAL_LOG_DIR / f"06_pipe2_{RUN_TS}.txt"


def period_root(period: str) -> Path:
    return ensure_dir(WRITE_ROOT / period)


for period in PERIODS:
    ensure_dir(period_root(period) / "indices")
    ensure_dir(period_root(period) / "tables")
    ensure_dir(period_root(period) / "reports")

ensure_dir(WRITE_ROOT / "tables")
ensure_dir(WRITE_ROOT / "reports")


def elapsed_seconds() -> float:
    return time.time() - PIPE_START_TS


def fmt_seconds(seconds: float) -> str:
    seconds = int(round(seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"


def log(message: str) -> None:
    now = datetime.now().strftime("%H:%M:%S")
    prefix = f"[{now} | +{fmt_seconds(elapsed_seconds())}]"
    line = f"{prefix} {message}"
    print(line, flush=True)
    GLOBAL_LOG_FILE.parent.mkdir(parents=True, exist_ok=True)
    with open(GLOBAL_LOG_FILE, "a", encoding="utf-8") as fh:
        fh.write(line + "\n")


def stage_banner(title: str) -> None:
    bar = "=" * 96
    log(bar)
    log(title)
    log(bar)


def period_input_csv(period: str) -> Path:
    return DATA_ROOT / TARGET_CORPUS / "04_rebuild_outputs" / READ_STAGE_CONSOL / f"{TARGET_CORPUS}_{period}_bibliometrix_clean.csv"


print("WRITE_ROOT =", WRITE_ROOT)
print("GLOBAL_LOG_FILE =", GLOBAL_LOG_FILE)


In [ ]:
# ============================================================
# Reconstrucao de docs_master e index_map por periodo
# ============================================================

EXPECTED_COLS = ["TI", "AB", "DE", "ID", "SO", "PY", "TC", "DI", "publication_date"]


def clean_text(value):
    if value is None:
        return pd.NA
    try:
        if pd.isna(value):
            return pd.NA
    except Exception:
        pass
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "<na>"}:
        return pd.NA
    return text


def normalize_text(value):
    text = clean_text(value)
    if pd.isna(text):
        return ""
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text


def stable_doc_id(row: pd.Series, period: str) -> str:
    seed = "|".join(
        [
            period,
            normalize_text(row.get("TI")),
            normalize_text(row.get("DI")),
            normalize_text(row.get("PY")),
            normalize_text(row.get("SO")),
        ]
    )
    return hashlib.md5(seed.encode("utf-8")).hexdigest()[:16]


def ensure_expected_cols(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in EXPECTED_COLS:
        if col not in out.columns:
            out[col] = pd.NA
    return out


def build_docs_master(period: str) -> pd.DataFrame:
    csv_path = period_input_csv(period)
    assert csv_path.exists(), f"Input nao encontrado: {csv_path}"
    raw = pd.read_csv(csv_path, dtype=str, low_memory=False)
    raw = ensure_expected_cols(raw)
    raw["period"] = period
    raw["title"] = raw["TI"].map(normalize_text)
    raw["abstract"] = raw["AB"].map(normalize_text)
    raw["keywords"] = (raw["DE"].fillna("").astype(str) + "; " + raw["ID"].fillna("").astype(str)).map(normalize_text)
    raw["source"] = raw["SO"].map(normalize_text)
    raw["year"] = pd.to_numeric(raw["PY"], errors="coerce").astype("Int64")
    raw["citations"] = pd.to_numeric(raw["TC"], errors="coerce").fillna(0)
    raw["doi"] = raw["DI"].map(normalize_text)
    raw["text_full"] = (
        raw["title"].fillna("").astype(str)
        + " [SEP] "
        + raw["abstract"].fillna("").astype(str)
        + " [SEP] "
        + raw["keywords"].fillna("").astype(str)
    ).str.replace(r"\s+", " ", regex=True).str.strip()
    raw["doc_id"] = raw.apply(lambda row: stable_doc_id(row, period), axis=1)
    raw["faiss_pos"] = np.arange(len(raw))

    docs_master = raw[
        ["doc_id", "faiss_pos", "period", "title", "abstract", "keywords", "source", "year", "citations", "doi", "publication_date", "text_full"]
    ].copy()
    docs_master = docs_master.drop_duplicates(subset=["doc_id"]).reset_index(drop=True)
    docs_master["faiss_pos"] = np.arange(len(docs_master))

    log(
        f"[docs] {period} | rows={len(docs_master)} | "
        f"abstract_pct={round(docs_master['abstract'].astype(str).str.len().gt(0).mean() * 100, 2) if len(docs_master) else 0.0}%"
    )
    return docs_master


stage_banner("RECONSTRUCAO DO CORPUS DE APLICACAO")

period_docs = {}
coverage_rows = []
for period in PERIODS:
    docs = build_docs_master(period)
    docs.to_csv(period_root(period) / "tables" / f"{TARGET_CORPUS}_{period}_docs_master.csv", index=False)
    docs[["faiss_pos", "doc_id", "period"]].to_csv(period_root(period) / "tables" / f"{TARGET_CORPUS}_{period}_index_map.csv", index=False)
    period_docs[period] = docs
    coverage_rows.append(
        {
            "period": period,
            "rows": int(len(docs)),
            "abstract_present_pct": round(docs["abstract"].astype(str).str.len().gt(0).mean() * 100, 2) if len(docs) else 0.0,
            "doi_present_pct": round(docs["doi"].astype(str).str.len().gt(0).mean() * 100, 2) if len(docs) else 0.0,
            "mean_year": round(float(docs["year"].dropna().astype(float).mean()), 2) if docs["year"].notna().any() else None,
        }
    )

coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv(WRITE_ROOT / "tables" / "ml_docs_master_coverage.csv", index=False)
display(coverage_df)


In [ ]:
# ============================================================
# Embeddings e indices FAISS por periodo
# ============================================================

stage_banner("EMBEDDINGS E INDICES")

device = "cuda" if torch.cuda.is_available() else "cpu"
generic_model = SentenceTransformer(GENERIC_MODEL_ID, device=device)
specialized_model = SentenceTransformer(str(MODEL_ROOT), device=device)
log(f"[model] generic={GENERIC_MODEL_ID} | specialized={MODEL_ROOT} | device={device}")


def encode_texts(model, texts: list[str], batch_size: int, period: str, label: str) -> np.ndarray:
    all_rows = []
    total_batches = math.ceil(len(texts) / batch_size) if texts else 0
    for batch_idx, start in enumerate(range(0, len(texts), batch_size), start=1):
        end = min(start + batch_size, len(texts))
        batch = texts[start:end]
        emb = model.encode(batch, convert_to_numpy=True, show_progress_bar=False).astype("float32")
        all_rows.append(emb)
        if batch_idx == 1 or batch_idx % 25 == 0 or batch_idx == total_batches:
            log(f"[embed] {period} | {label} | batch {batch_idx}/{total_batches} | rows {end}/{len(texts)}")
    return np.vstack(all_rows) if all_rows else np.zeros((0, 768), dtype="float32")


embedding_manifest_rows = []
for period in PERIODS:
    docs = period_docs[period]
    texts = docs["text_full"].fillna("").astype(str).tolist()
    for label, model in [("generic", generic_model), ("specialized", specialized_model)]:
        emb = encode_texts(model, texts, EMBED_BATCH_SIZE, period, label)
        faiss.normalize_L2(emb)
        index = faiss.IndexFlatIP(emb.shape[1])
        index.add(emb)

        np.save(period_root(period) / "indices" / f"{TARGET_CORPUS}_{period}_{label}_embeddings.npy", emb)
        faiss.write_index(index, str(period_root(period) / "indices" / f"{TARGET_CORPUS}_{period}_{label}.faiss"))

        embedding_manifest_rows.append(
            {
                "period": period,
                "retriever": label,
                "rows": int(len(docs)),
                "embedding_dim": int(emb.shape[1]) if emb.size else 0,
                "index_ntotal": int(index.ntotal),
            }
        )

embedding_manifest_df = pd.DataFrame(embedding_manifest_rows)
embedding_manifest_df.to_csv(WRITE_ROOT / "tables" / "embedding_index_manifest.csv", index=False)
display(embedding_manifest_df)


In [ ]:
# ============================================================
# Banco de queries derivado apenas do core
# ============================================================

stage_banner("BANCO DE QUERIES DO CORE")


def parse_keywords(value) -> list[str]:
    if value is None:
        return []
    if isinstance(value, list):
        return [str(x) for x in value if str(x).strip()]
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(x) for x in parsed if str(x).strip()]
    except Exception:
        pass
    text = text.replace("[", "").replace("]", "").replace("'", "").replace('"', "")
    return [part.strip() for part in text.split(",") if part.strip()]


query_rows = []
for corpus in QUERY_CORPORA:
    cards_path = DATA_ROOT / corpus / "04_rebuild_outputs" / READ_STAGE_ABSTRACT / "core" / "tables" / f"{corpus}_core_topic_cards.csv"
    assert cards_path.exists(), f"Topic cards core nao encontrados: {cards_path}"
    cards = pd.read_csv(cards_path).head(TOPICS_PER_CORPUS).copy()
    log(f"[query-bank] {corpus} | rows={len(cards)} | path={cards_path}")
    for row in cards.itertuples(index=False):
        keywords = parse_keywords(getattr(row, "keywords", ""))
        title = str(getattr(row, "title", "")).strip()
        query_text = (title + ". " + " ".join(keywords[:6])).strip()
        query_text = re.sub(r"\s+", " ", query_text)
        if len(query_text.split()) < 3:
            continue
        query_rows.append(
            {
                "query_id": f"{corpus}_topic_{getattr(row, 'topic', 'na')}",
                "source_corpus": corpus,
                "topic": getattr(row, "topic", None),
                "query_text": query_text,
                "topic_size": getattr(row, "size", None),
                "keywords": "; ".join(keywords[:10]),
            }
        )

query_bank = pd.DataFrame(query_rows).drop_duplicates(subset=["query_id", "query_text"]).reset_index(drop=True)
query_bank.to_csv(WRITE_ROOT / "tables" / "core_gap_query_bank.csv", index=False)

log(f"[query-bank] total queries={len(query_bank)}")
display(query_bank.head(20))


In [ ]:
# ============================================================
# Analytics de recuperacao e candidatos de white space
# ============================================================

stage_banner("ANALYTICS DE RECUPERACAO")

assets = {}
for period in PERIODS:
    docs = pd.read_csv(period_root(period) / "tables" / f"{TARGET_CORPUS}_{period}_docs_master.csv")
    idx_map = pd.read_csv(period_root(period) / "tables" / f"{TARGET_CORPUS}_{period}_index_map.csv")
    assets[period] = {
        "docs": docs,
        "idx_map": idx_map,
        "generic": faiss.read_index(str(period_root(period) / "indices" / f"{TARGET_CORPUS}_{period}_generic.faiss")),
        "specialized": faiss.read_index(str(period_root(period) / "indices" / f"{TARGET_CORPUS}_{period}_specialized.faiss")),
    }


def semantic_search(model, index, docs: pd.DataFrame, query: str, topk: int, period: str, label: str) -> pd.DataFrame:
    q_emb = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)
    scores, idxs = index.search(q_emb, topk)
    hits = docs.iloc[idxs[0]].copy().reset_index(drop=True)
    hits["rank"] = np.arange(1, len(hits) + 1)
    hits["score"] = scores[0]
    hits["period"] = period
    hits["retriever"] = label
    return hits


semantic_hit_rows = []
metric_rows = []

for period in PERIODS:
    log(f"[analytics] iniciando periodo={period}")
    docs = assets[period]["docs"]
    total_queries = len(query_bank)
    for q_idx, q_row in enumerate(query_bank.itertuples(index=False), start=1):
        gen_hits = semantic_search(generic_model, assets[period]["generic"], docs, q_row.query_text, TOPK, period, "generic")
        spec_hits = semantic_search(specialized_model, assets[period]["specialized"], docs, q_row.query_text, TOPK, period, "specialized")

        gen_ids = set(gen_hits["doc_id"].head(10).tolist())
        spec_ids = set(spec_hits["doc_id"].head(10).tolist())
        overlap10 = len(gen_ids & spec_ids) / max(1, len(gen_ids | spec_ids))
        metric_rows.append(
            {
                "period": period,
                "query_id": q_row.query_id,
                "source_corpus": q_row.source_corpus,
                "query_text": q_row.query_text,
                "generic_top1_score": float(gen_hits["score"].iloc[0]),
                "specialized_top1_score": float(spec_hits["score"].iloc[0]),
                "score_lift_top1": float(spec_hits["score"].iloc[0] - gen_hits["score"].iloc[0]),
                "overlap_at_10": round(float(overlap10), 4),
                "specialized_mean_citations_top10": round(float(pd.to_numeric(spec_hits["citations"], errors="coerce").fillna(0).head(10).mean()), 2),
                "specialized_recent_share_top10": round(float(pd.to_numeric(spec_hits["year"], errors="coerce").fillna(0).head(10).ge(2023).mean()), 4),
            }
        )

        for hit_df in [gen_hits, spec_hits]:
            hit_df = hit_df.copy()
            hit_df["query_id"] = q_row.query_id
            hit_df["query_text"] = q_row.query_text
            hit_df["source_corpus"] = q_row.source_corpus
            semantic_hit_rows.append(hit_df)

        if q_idx == 1 or q_idx % 10 == 0 or q_idx == total_queries:
            log(f"[analytics] {period} | query {q_idx}/{total_queries}")

semantic_hits = pd.concat(semantic_hit_rows, ignore_index=True)
semantic_metrics = pd.DataFrame(metric_rows)
semantic_hits.to_csv(WRITE_ROOT / "tables" / "semantic_query_hits.csv", index=False)
semantic_metrics.to_csv(WRITE_ROOT / "tables" / "semantic_query_metrics.csv", index=False)

white_space = semantic_metrics.copy()
white_space["candidate_score"] = white_space["score_lift_top1"] + (1 - white_space["overlap_at_10"]) * 0.25
white_space = white_space.sort_values(["period", "candidate_score"], ascending=[True, False]).reset_index(drop=True)
white_space.to_csv(WRITE_ROOT / "tables" / "white_space_candidates.csv", index=False)

display(semantic_metrics.head(20))
display(white_space.head(20))


In [ ]:
# ============================================================
# Manifesto final da camada 2
# ============================================================

stage_banner("MANIFESTO FINAL DA CAMADA 2")

manifest_rows = []
for path in sorted(WRITE_ROOT.rglob("*")):
    if path.is_file():
        manifest_rows.append(
            {
                "artifact": str(path),
                "size_bytes": path.stat().st_size,
            }
        )

manifest_df = pd.DataFrame(manifest_rows)
manifest_df.to_csv(WRITE_ROOT / "reports" / "artifact_manifest.csv", index=False)

display(pd.read_csv(WRITE_ROOT / "tables" / "ml_docs_master_coverage.csv"))
display(pd.read_csv(WRITE_ROOT / "tables" / "embedding_index_manifest.csv"))
display(manifest_df.head(30))

print("Arquivos finais salvos em:", WRITE_ROOT)
